In [1]:
import pandas as pd
from pathlib import Path

def tim_file(ten_file):
    cac_duong_dan = [
        Path(ten_file),
        Path("data") / ten_file,
        Path("..") / "data" / ten_file
    ]

    for duong_dan in cac_duong_dan:
        if duong_dan.is_file():
            return duong_dan

    raise FileNotFoundError(f"Khong tim thay file {ten_file}")

# Đọc dữ liệu sinh viên
duong_dan_sinh_vien = tim_file("sinh_vien.csv")
df_sinh_vien = pd.read_csv(duong_dan_sinh_vien)

# Đọc dữ liệu tin nhắn
duong_dan_tin_nhan = tim_file("tin_nhan.csv")
df_tin_nhan = pd.read_csv(duong_dan_tin_nhan)

print("Da doc du lieu thanh cong!")
print("Sinh vien:", duong_dan_sinh_vien, df_sinh_vien.shape)
print("Tin nhan:", duong_dan_tin_nhan, df_tin_nhan.shape)


Da doc du lieu thanh cong!
Sinh vien: data\sinh_vien.csv (120, 3)
Tin nhan: data\tin_nhan.csv (300, 2)


# Bài tập 1: Bayes với ba nguyên nhân

In [2]:

# -*- coding: utf-8 -*-
"""Bài tập 1: Bayes với ba nhóm sinh viên."""

def bayes_nhieu_nhom(tien_nghiem, kha_nang):
    """Tính xác suất hậu nghiệm cho nhiều nhóm."""
    tu_so = [
        p * k
        for p, k in zip(tien_nghiem, kha_nang)
    ]
    mau_so = sum(tu_so)
    return [tu / mau_so for tu in tu_so]


# Xác suất ban đầu của từng nhóm
tien_nghiem = [0.50, 0.30, 0.20]

# Xác suất qua môn của từng nhóm
kha_nang = [0.95, 0.70, 0.30]

# Tính xác suất thuộc từng nhóm khi biết đã qua môn
hau_nghiem = bayes_nhieu_nhom(tien_nghiem, kha_nang)

ten_nhom = ["Hoc deu", "Hoc don", "It hoc"]

print("Xac suat sinh vien thuoc tung nhom khi da qua mon:")
for ten, p in zip(ten_nhom, hau_nghiem):
    print(f" {ten}: {p:.4f}")

print()
print(f"Tong ba xac suat: {sum(hau_nghiem):.4f}")

# Kiem tra tong cac xac suat hau nghiem bang 1
assert abs(sum(hau_nghiem) - 1.0) < 1e-9
print("Kiem tra: Tong cac xac suat bang 1.")
print("\nNhận xét: Trong nhóm sinh viên đã qua môn, nhóm Học đều chiếm tỷ lệ cao nhất với xác suất 0,6376 (63,76%), tiếp theo là nhóm Học dồn với 0,2819 (28,19%). \nNhóm Ít học chiếm tỷ lệ thấp nhất, chỉ 0,0805 (8,05%). Tổng ba xác suất bằng 1 (100%), cho thấy các xác suất đã được chuẩn hóa chính xác. \nKết quả cho thấy sinh viên học đều chiếm phần lớn trong nhóm sinh viên đã qua môn.")

Xac suat sinh vien thuoc tung nhom khi da qua mon:
 Hoc deu: 0.6376
 Hoc don: 0.2819
 It hoc: 0.0805

Tong ba xac suat: 1.0000
Kiem tra: Tong cac xac suat bang 1.

Nhận xét: Trong nhóm sinh viên đã qua môn, nhóm Học đều chiếm tỷ lệ cao nhất với xác suất 0,6376 (63,76%), tiếp theo là nhóm Học dồn với 0,2819 (28,19%). 
Nhóm Ít học chiếm tỷ lệ thấp nhất, chỉ 0,0805 (8,05%). Tổng ba xác suất bằng 1 (100%), cho thấy các xác suất đã được chuẩn hóa chính xác. 
Kết quả cho thấy sinh viên học đều chiếm phần lớn trong nhóm sinh viên đã qua môn.


# Bài tập 2: Từ nào tố cáo mạnh nhất

In [3]:

# -*- coding: utf-8 -*-
"""Bài tập 2: Tìm từ tố cáo tin rác mạnh nhất."""

import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# Đọc dữ liệu tin nhắn
if "df_tin_nhan" in globals():
    df = df_tin_nhan
else:
    cac_duong_dan = [
        Path("tin_nhan.csv"),
        Path("data") / "tin_nhan.csv",
        Path("..") / "data" / "tin_nhan.csv"
    ]

    duong_dan_du_lieu = next(
        (p for p in cac_duong_dan if p.is_file()), None
    )

    if duong_dan_du_lieu is None:
        raise FileNotFoundError("Khong tim thay file tin_nhan.csv")

    df = pd.read_csv(duong_dan_du_lieu)

print("So tin nhan:", len(df))
print()

# Biến văn bản thành bảng đếm từ
vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"].fillna("").astype(str))
y = df["la_rac"]

# Huấn luyện trên toàn bộ dữ liệu
mo_hinh = MultinomialNB()
mo_hinh.fit(X, y)

# Tính chênh lệch log xác suất giữa lớp rác và lớp thường
tu_dien = vec.get_feature_names_out()
hieu = mo_hinh.feature_log_prob_[1] - mo_hinh.feature_log_prob_[0]

# Năm từ nghiêng mạnh nhất về lớp tin rác
top_rac = hieu.argsort()[::-1][:5]

print("5 tu to cao tin rac manh nhat:")
for i in top_rac:
    print(f" {tu_dien[i]:15s} {hieu[i]:.4f}")

print()

# Năm từ nghiêng mạnh nhất về lớp tin thường
top_thuong = hieu.argsort()[:5]

print("5 tu cho thay tin thuong manh nhat:")
for i in top_thuong:
    print(f" {tu_dien[i]:15s} {hieu[i]:.4f}")
print("\nNhận xét:")
print("Trong 300 tin nhắn, các từ “mãi”, “tay”, “ngay”, “miễn” và “trúng” có điểm số cao nhất cho thấy khả năng xuất hiện trong tin nhắn rác, trong đó từ “mãi” có điểm số cao nhất là 3.2034.")
print("Ngược lại, các từ “lớp”, “em”, “bài”, “báo” và “phòng” có điểm số âm, nghiêng về nhóm tin nhắn thông thường; từ “lớp” có điểm số thấp nhất là -2.6122.")
print("Kết quả cho thấy các từ liên quan đến nội dung quảng cáo, ưu đãi có thể là dấu hiệu nhận biết tin nhắn rác, trong khi các từ thường dùng trong giao tiếp học tập có xu hướng xuất hiện trong tin nhắn thông thường.")
print("Tuy nhiên, cần kết hợp nhiều từ và ngữ cảnh để phân loại chính xác hơn.")

So tin nhan: 300

5 tu to cao tin rac manh nhat:
 mãi             3.2034
 tay             2.9056
 ngay            2.6618
 miễn            2.4795
 trúng           2.3742

5 tu cho thay tin thuong manh nhat:
 lớp             -2.6122
 em              -2.5618
 bài             -2.2946
 báo             -2.2741
 phòng           -2.2741

Nhận xét:
Trong 300 tin nhắn, các từ “mãi”, “tay”, “ngay”, “miễn” và “trúng” có điểm số cao nhất cho thấy khả năng xuất hiện trong tin nhắn rác, trong đó từ “mãi” có điểm số cao nhất là 3.2034.
Ngược lại, các từ “lớp”, “em”, “bài”, “báo” và “phòng” có điểm số âm, nghiêng về nhóm tin nhắn thông thường; từ “lớp” có điểm số thấp nhất là -2.6122.
Kết quả cho thấy các từ liên quan đến nội dung quảng cáo, ưu đãi có thể là dấu hiệu nhận biết tin nhắn rác, trong khi các từ thường dùng trong giao tiếp học tập có xu hướng xuất hiện trong tin nhắn thông thường.
Tuy nhiên, cần kết hợp nhiều từ và ngữ cảnh để phân loại chính xác hơn.


# Bài tập 3: Đổi tham số làm trơn

In [4]:

# -*- coding: utf-8 -*-
"""Bài tập 3: Đổi tham số làm trơn."""

import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# Đọc dữ liệu tin nhắn
if "df_tin_nhan" in globals():
    df = df_tin_nhan
else:
    cac_duong_dan = [
        Path("tin_nhan.csv"),
        Path("data") / "tin_nhan.csv",
        Path("..") / "data" / "tin_nhan.csv"
    ]
    duong_dan_du_lieu = next(
        (p for p in cac_duong_dan if p.is_file()), None
    )
    if duong_dan_du_lieu is None:
        raise FileNotFoundError("Khong tim thay file tin_nhan.csv")
    df = pd.read_csv(duong_dan_du_lieu)

# Chuyển văn bản thành dữ liệu số
vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"].fillna("").astype(str))
y = df["la_rac"]

# Chia dữ liệu cố định để so sánh các giá trị alpha
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# Thử các giá trị alpha khác nhau
cac_alpha = [0.001, 0.1, 1.0, 10.0, 100.0]

print("Ket qua voi cac gia tri alpha:")
print(f"{'Alpha':>8s} {'Accuracy':>12s} {'Precision':>12s} {'Recall':>12s}")

for alpha in cac_alpha:
    mo_hinh = MultinomialNB(alpha=alpha)
    mo_hinh.fit(X_train, y_train)
    y_pred = mo_hinh.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)

    print(f"{alpha:8.3f} {acc:12.4f} {pre:12.4f} {rec:12.4f}")
print("\nNhận xét:")
print("Khi thay đổi giá trị alpha từ 0.001 đến 100, các chỉ số Accuracy, Precision và Recall đều không thay đổi, lần lượt đạt 0.9467, 1.0000 và 0.8667.")
print("Điều này cho thấy mô hình Naive Bayes hoạt động ổn định với các giá trị alpha được thử nghiệm trên tập dữ liệu này.")
print("Precision đạt 100% cho thấy các tin nhắn được dự đoán là tin rác đều thực sự là tin rác trong tập kiểm tra.")
print("Tuy nhiên, Recall đạt 86,67% cho thấy mô hình vẫn bỏ sót một số tin nhắn rác. Nhìn chung, mô hình có độ chính xác cao, nhưng cần đánh giá thêm với các tập dữ liệu khác để xác định giá trị alpha phù hợp nhất.")


Ket qua voi cac gia tri alpha:
   Alpha     Accuracy    Precision       Recall
   0.001       0.9467       1.0000       0.8667
   0.100       0.9467       1.0000       0.8667
   1.000       0.9467       1.0000       0.8667
  10.000       0.9467       1.0000       0.8667
 100.000       0.9467       1.0000       0.8667

Nhận xét:
Khi thay đổi giá trị alpha từ 0.001 đến 100, các chỉ số Accuracy, Precision và Recall đều không thay đổi, lần lượt đạt 0.9467, 1.0000 và 0.8667.
Điều này cho thấy mô hình Naive Bayes hoạt động ổn định với các giá trị alpha được thử nghiệm trên tập dữ liệu này.
Precision đạt 100% cho thấy các tin nhắn được dự đoán là tin rác đều thực sự là tin rác trong tập kiểm tra.
Tuy nhiên, Recall đạt 86,67% cho thấy mô hình vẫn bỏ sót một số tin nhắn rác. Nhìn chung, mô hình có độ chính xác cao, nhưng cần đánh giá thêm với các tập dữ liệu khác để xác định giá trị alpha phù hợp nhất.


# Bài tập 4: Tự viết bộ lọc không dùng thư viện

In [5]:
# -*- coding: utf-8 -*-
"""Bài 4: tự đếm từ và tính log điểm Naive Bayes trên đủ 300 tin."""

from collections import Counter
from math import log

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB


df = pd.read_csv("data/tin_nhan.csv")
tin_train, tin_test, y_train, y_test = train_test_split(
    df["noi_dung"], df["la_rac"], test_size=0.25, random_state=17,
    stratify=df["la_rac"],
)

# Cùng quy tắc r"(?u)\S+" của CountVectorizer: chuyển chữ thường, tách khoảng trắng.
def tach_tu(cau):
    return cau.lower().split()


tu_dien = {tu for cau in tin_train for tu in tach_tu(cau)}
V = len(tu_dien)
dem_lop = Counter(y_train)
dem_tu = {0: Counter(), 1: Counter()}
for cau, nhan in zip(tin_train, y_train):
    dem_tu[int(nhan)].update(tach_tu(cau))
tong_tu = {nhan: sum(dem_tu[nhan].values()) for nhan in [0, 1]}


def diem_log(cau, nhan, alpha=1.0):
    """log P(nhãn) + tổng số lần từ xuất hiện nhân log P(từ|nhãn)."""
    diem = log(dem_lop[nhan] / len(y_train))
    for tu, so_lan in Counter(tach_tu(cau)).items():
        if tu in tu_dien:  # Từ mới bị bỏ qua, giống vec.transform.
            xac_suat = (dem_tu[nhan][tu] + alpha) / (tong_tu[nhan] + alpha * V)
            diem += so_lan * log(xac_suat)
    return diem


du_doan_tu_viet = [int(diem_log(cau, 1) > diem_log(cau, 0)) for cau in tin_test]
acc_tu_viet = accuracy_score(y_test, du_doan_tu_viet)

vec = CountVectorizer(token_pattern=r"(?u)\S+")
X_train = vec.fit_transform(tin_train)
X_test = vec.transform(tin_test)
mo_hinh = MultinomialNB(alpha=1.0).fit(X_train, y_train)
du_doan_thu_vien = mo_hinh.predict(X_test)
acc_thu_vien = accuracy_score(y_test, du_doan_thu_vien)
so_du_doan_khac = sum(a != b for a, b in zip(du_doan_tu_viet, du_doan_thu_vien))

print(f"So mau train/test: {len(tin_train)}/{len(tin_test)}")
print(f"So tu trong tu dien tu viet: {V}")
print(f"So tu trong tu dien thu vien: {len(vec.vocabulary_)}")
print(f"Accuracy tu viet: {acc_tu_viet:.4f}")
print(f"Accuracy MultinomialNB: {acc_thu_vien:.4f}")
print(f"So du doan khac nhau: {so_du_doan_khac}")

print("\nGiải thích: Hai phương pháp sử dụng cùng tập dữ liệu huấn luyện và kiểm tra,")
print("cùng quy tắc tách từ, xác suất tiên nghiệm và giá trị alpha = 1.")
print("Cả hai đều bỏ qua các từ không có trong từ điển.")
print("Việc cộng log xác suất giúp tránh hiện tượng tràn số dưới khi xử lý tin nhắn dài.")

print("\nNhận xét: Bộ lọc tự viết và mô hình MultinomialNB đều đạt Accuracy = 0,9467 (94,67%), với số lượng từ trong từ điển của cả hai phương pháp đều là 40.")
print("Hai mô hình không có dự đoán khác nhau trên tập kiểm tra gồm 75 tin nhắn.")
print("Kết quả cho thấy bộ lọc tự viết đã triển khai chính xác các bước đếm từ, làm trơn Laplace với \(\alpha = 1\), tính xác suất tiên nghiệm và cộng log xác suất.")
print("Việc sử dụng tổng log xác suất giúp hạn chế hiện tượng tràn số dưới khi xử lý tin nhắn dài.")
print("Như vậy, kết quả của bài tập chứng minh cách cài đặt thủ công hoạt động tương đương với MultinomialNB trên tập dữ liệu đang xét.")


So mau train/test: 225/75
So tu trong tu dien tu viet: 40
So tu trong tu dien thu vien: 40
Accuracy tu viet: 0.9467
Accuracy MultinomialNB: 0.9467
So du doan khac nhau: 0

Giải thích: Hai phương pháp sử dụng cùng tập dữ liệu huấn luyện và kiểm tra,
cùng quy tắc tách từ, xác suất tiên nghiệm và giá trị alpha = 1.
Cả hai đều bỏ qua các từ không có trong từ điển.
Việc cộng log xác suất giúp tránh hiện tượng tràn số dưới khi xử lý tin nhắn dài.

Nhận xét: Bộ lọc tự viết và mô hình MultinomialNB đều đạt Accuracy = 0,9467 (94,67%), với số lượng từ trong từ điển của cả hai phương pháp đều là 40.
Hai mô hình không có dự đoán khác nhau trên tập kiểm tra gồm 75 tin nhắn.
Kết quả cho thấy bộ lọc tự viết đã triển khai chính xác các bước đếm từ, làm trơn Laplace với \(lpha = 1\), tính xác suất tiên nghiệm và cộng log xác suất.
Việc sử dụng tổng log xác suất giúp hạn chế hiện tượng tràn số dưới khi xử lý tin nhắn dài.
Như vậy, kết quả của bài tập chứng minh cách cài đặt thủ công hoạt động tương đươ

<>:69: SyntaxWarning: invalid escape sequence '\('
<>:69: SyntaxWarning: invalid escape sequence '\('
C:\Users\Admin\AppData\Local\Temp\ipykernel_5472\386455632.py:69: SyntaxWarning: invalid escape sequence '\('
  print("Kết quả cho thấy bộ lọc tự viết đã triển khai chính xác các bước đếm từ, làm trơn Laplace với \(\alpha = 1\), tính xác suất tiên nghiệm và cộng log xác suất.")


# Bài tập 5: Chọn sai biến thể Naive Bayes

In [6]:
# -*- coding: utf-8 -*-
"""Bài 5: cố ý dùng sai MultinomialNB cho hai đặc trưng liên tục."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB


df = pd.read_csv("data/sinh_vien.csv")
X = df[["gio_on", "diem_giua_ky"]]
y = df["qua_mon"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y,
)
for ten, mo_hinh in [
    ("MultinomialNB", MultinomialNB()),
    ("GaussianNB", GaussianNB()),
]:
    mo_hinh.fit(X_train, y_train)
    print(f"{ten}: accuracy = {accuracy_score(y_test, mo_hinh.predict(X_test)):.4f}")

print("\nNhận xét:")
print("Mô hình MultinomialNB đạt Accuracy = 0,8000 (80%), trong khi mô hình GaussianNB đạt Accuracy = 0,9000 (90%), cao hơn 10 điểm phần trăm.")
print("Kết quả cho thấy GaussianNB hoạt động hiệu quả hơn trên bộ dữ liệu sinh viên với hai đặc trưng gio_on và diem_giua_ky.")
print("Nguyên nhân là MultinomialNB phù hợp với dữ liệu đếm tần suất, còn GaussianNB phù hợp hơn với các đặc trưng số liên tục.")
print("Vì vậy, dù MultinomialNB vẫn chạy được, việc sử dụng mô hình này không phù hợp về mặt nguyên tắc với bản chất dữ liệu của bài toán.")

MultinomialNB: accuracy = 0.8000
GaussianNB: accuracy = 0.9000

Nhận xét:
Mô hình MultinomialNB đạt Accuracy = 0,8000 (80%), trong khi mô hình GaussianNB đạt Accuracy = 0,9000 (90%), cao hơn 10 điểm phần trăm.
Kết quả cho thấy GaussianNB hoạt động hiệu quả hơn trên bộ dữ liệu sinh viên với hai đặc trưng gio_on và diem_giua_ky.
Nguyên nhân là MultinomialNB phù hợp với dữ liệu đếm tần suất, còn GaussianNB phù hợp hơn với các đặc trưng số liên tục.
Vì vậy, dù MultinomialNB vẫn chạy được, việc sử dụng mô hình này không phù hợp về mặt nguyên tắc với bản chất dữ liệu của bài toán.


# Bài tập 6: Lớp hiếm và xác suất tiên nghiệm

In [7]:
# -*- coding: utf-8 -*-
"""Bài 6: thay đổi prior bằng cách chỉ giữ 20 tin rác đầu tiên."""

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB


df = pd.read_csv("data/tin_nhan.csv")
bo_moi = pd.concat([
    df.loc[df["la_rac"] == 0],
    df.loc[df["la_rac"] == 1].head(20),
], ignore_index=True)
tin_train, tin_test, y_train, y_test = train_test_split(
    bo_moi["noi_dung"], bo_moi["la_rac"], test_size=0.25,
    random_state=17, stratify=bo_moi["la_rac"],
)
vec = CountVectorizer(token_pattern=r"(?u)\S+")
X_train = vec.fit_transform(tin_train)
X_test = vec.transform(tin_test)
mo_hinh = MultinomialNB(alpha=1.0).fit(X_train, y_train)
y_pred = mo_hinh.predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

print("So tin tung lop trong bo du lieu moi:")
print(bo_moi["la_rac"].value_counts().sort_index())
print(f"Prior lop rac trong tap hoc = {mo_hinh.class_count_[1] / sum(mo_hinh.class_count_):.4f}")
print(f"Ma tran: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"Accuracy = {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall lop rac = {recall_score(y_test, y_pred, pos_label=1):.4f}")
print("Moc doan tat ca la thuong: accuracy = 0.9000, recall rac = 0.0000")

print("\nNhận xét:")
print("Bộ lọc đạt Accuracy = 0,9800 (98%) và Recall của lớp rác = 0,8000 (80%).")
print("Điều này cho thấy mô hình phân loại đúng 98% tổng số tin nhắn và phát hiện được 80% tin rác thực tế, tuy nhiên vẫn bỏ sót 1 tin rác (FN = 1).")
print("Đáng chú ý, nếu dự đoán tất cả tin nhắn đều là tin thường, mô hình vẫn đạt Accuracy = 90% do dữ liệu mất cân bằng, nhưng Recall của lớp rác bằng 0%.")
print("Vì vậy, Accuracy không đủ để đánh giá chất lượng bộ lọc khi lớp rác chiếm tỷ lệ nhỏ. Cần kết hợp Recall và số lượng tin rác bị bỏ sót (FN) để đánh giá khả năng phát hiện tin rác của mô hình.")

So tin tung lop trong bo du lieu moi:
la_rac
0    180
1     20
Name: count, dtype: int64
Prior lop rac trong tap hoc = 0.1000
Ma tran: TN=45, FP=0, FN=1, TP=4
Accuracy = 0.9800
Recall lop rac = 0.8000
Moc doan tat ca la thuong: accuracy = 0.9000, recall rac = 0.0000

Nhận xét:
Bộ lọc đạt Accuracy = 0,9800 (98%) và Recall của lớp rác = 0,8000 (80%).
Điều này cho thấy mô hình phân loại đúng 98% tổng số tin nhắn và phát hiện được 80% tin rác thực tế, tuy nhiên vẫn bỏ sót 1 tin rác (FN = 1).
Đáng chú ý, nếu dự đoán tất cả tin nhắn đều là tin thường, mô hình vẫn đạt Accuracy = 90% do dữ liệu mất cân bằng, nhưng Recall của lớp rác bằng 0%.
Vì vậy, Accuracy không đủ để đánh giá chất lượng bộ lọc khi lớp rác chiếm tỷ lệ nhỏ. Cần kết hợp Recall và số lượng tin rác bị bỏ sót (FN) để đánh giá khả năng phát hiện tin rác của mô hình.
